# Solving Pickle Doomsday

In [13]:
from oggm import cfg, utils
from oggm import workflow
from oggm import DEFAULT_BASE_URL
import oggmzarr.datacube.convert as convert

In [2]:
cfg.initialize(logging_level="ERROR")

cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["continue_on_error"] = True
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='oggm-geozarr', reset=True)
rgi_ids = ['RGI60-11.00897', "RGI60-06.00377"] # HEF, Bruarjoekull 

DEFAULT_BASE_URL

2026-04-30 08:46:48: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-04-30 08:46:48: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-04-30 08:46:48: oggm.cfg: Multiprocessing: using all available processors (N=20)
2026-04-30 08:46:48: oggm.cfg: Multiprocessing switched ON after user settings.
2026-04-30 08:46:48: oggm.cfg: PARAMS['continue_on_error'] changed from `False` to `True`.


'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup/'

In [3]:
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    prepro_base_url=DEFAULT_BASE_URL,  # where to fetch the data?
    from_prepro_level=4,  # what kind of data? 
    prepro_border=80  # how big of a map?
)
gdir = gdirs[0]

2026-04-30 08:46:48: oggm.workflow: init_glacier_directories from prepro level 4 on 2 glaciers.
2026-04-30 08:46:48: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 2 glaciers


In [4]:
pickle_paths = convert.get_pickle_paths(gdir)
pickle_paths

[PosixPath('model_flowlines_dyn_melt_f_calib.pkl'),
 PosixPath('model_flowlines.pkl'),
 PosixPath('inversion_output.pkl'),
 PosixPath('inversion_input.pkl'),
 PosixPath('inversion_flowlines.pkl'),
 PosixPath('downstream_line.pkl')]

In [5]:
def text_to_dict(lines):
    v_text = ["| variable| type | description |\n|---|---|---|"]
    v_text.append(f"{"\n  - ".join([f"<b>{i}</b>: {str(j)[7:-1]}" for i,j in lines.items()])}")
    return v_text

In [6]:
pickle_data = convert.get_pickle_data(pickle_paths, gdir, type_only=False)
pickle_types = convert.get_pickle_data(pickle_paths, gdir, type_only=True)


model_flowlines_dyn_melt_f_calib not in cfg.BASENAMES.
Pickle model_flowlines_dyn_melt_f_calib of type <class 'str'> not parseable.
'Centerline' object has no attribute 'items'
Pickle downstream_line of type <class 'str'> not parseable.
model_flowlines_dyn_melt_f_calib not in cfg.BASENAMES.
Pickle model_flowlines_dyn_melt_f_calib of type <class 'str'> not parseable.
'Centerline' object has no attribute 'items'
Pickle downstream_line of type <class 'str'> not parseable.


Generate GH markdown for pickle types.

In [7]:
# pprint.pp(pickle_data, sort_dicts=True)
text = []
for k,v in pickle_types.items():
    if isinstance(v, list):
        if isinstance(v[0], dict):
            v_text = text_to_dict(v[0])
        else:
            v_text = f"{v}"
    elif isinstance(v, dict):
        v_text = text_to_dict(v)
    else:
        v_text = f"{v}"
    text.append(f"| <b>{k}</b> | {v_text}| |\n")
print("\n".join(text))

| <b>model_flowlines</b> | [<class 'oggm.core.flowline.MixedBedFlowline'>]| |

| <b>inversion_output</b> | ['| variable| type | description |\n|---|---|---|', "<b>dx</b>: 'float'\n  - <b>flux_a0</b>: 'numpy.ndarray'\n  - <b>width</b>: 'numpy.ndarray'\n  - <b>slope_angle</b>: 'numpy.ndarray'\n  - <b>is_rectangular</b>: 'numpy.ndarray'\n  - <b>is_trapezoid</b>: 'numpy.ndarray'\n  - <b>flux</b>: 'numpy.ndarray'\n  - <b>is_last</b>: 'bool'\n  - <b>hgt</b>: 'numpy.ndarray'\n  - <b>invert_with_trapezoid</b>: 'bool'\n  - <b>thick</b>: 'numpy.ndarray'\n  - <b>volume</b>: 'numpy.ndarray'"]| |

| <b>inversion_input</b> | ['| variable| type | description |\n|---|---|---|', "<b>dx</b>: 'float'\n  - <b>flux_a0</b>: 'numpy.ndarray'\n  - <b>width</b>: 'numpy.ndarray'\n  - <b>slope_angle</b>: 'numpy.ndarray'\n  - <b>is_rectangular</b>: 'numpy.ndarray'\n  - <b>is_trapezoid</b>: 'numpy.ndarray'\n  - <b>flux</b>: 'numpy.ndarray'\n  - <b>is_last</b>: 'bool'\n  - <b>hgt</b>: 'numpy.ndarray'\n  - <b>invert_

Convert supported pickles to data tree (currently only `inversion_output` and `inversion_input`)

In [8]:
data_tree = convert.convert_pickle_to_datatree(pickle_data=pickle_data)
data_tree

<xarray.DataTree>
Group: /
├── Group: /inversion_output
│       Dimensions:                (flux_a0: 50, width: 50, slope_angle: 50,
│                                   is_rectangular: 50, is_trapezoid: 50, flux: 50,
│                                   hgt: 50, thick: 50, volume: 50)
│       Coordinates:
│         * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
│         * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
│         * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
│         * is_rectangular         (is_rectangular) bool 50B False False ... False False
│         * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
│         * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
│         * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
│         * thick                  (thick) float64 400B 33.17 35.88 ... 38.42 27.32
│         * volume                 (volume) float64 400B 2.5e+06 2.068e+06 ... 1.257e+06
│       Data variables:
│           dx                     float64 8B 100.0
│           is_last                bool 1B True
│           invert_with_trapezoid  bool 1B True
└── Group: /inversion_input
        Dimensions:                (flux_a0: 50, width: 50, slope_angle: 50,
                                    is_rectangular: 50, is_trapezoid: 50, flux: 50,
                                    hgt: 50)
        Coordinates:
          * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
          * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
          * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
          * is_rectangular         (is_rectangular) bool 50B False False ... False False
          * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
          * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
          * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
        Data variables:
            dx                     float64 8B 100.0
            is_last                bool 1B True
            invert_with_trapezoid  bool 1B True

In [9]:
convert.write_zarr(data_tree=data_tree, storage_directory=f"{gdir.rgi_id}.zarr",)

Check we can read the generated zarr

In [12]:
stream_cube = xr.open_datatree(
    f"{gdir.rgi_id}.zarr",
    group=None,
    chunks={},
    engine="zarr",
    consolidated=True,
    decode_cf=True,
)
stream_cube

<xarray.DataTree>
Group: /
├── Group: /inversion_input
│       Dimensions:                (flux: 50, flux_a0: 50, hgt: 50, is_rectangular: 50,
│                                   is_trapezoid: 50, slope_angle: 50, width: 50)
│       Coordinates:
│         * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
│         * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
│         * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
│         * is_rectangular         (is_rectangular) bool 50B False False ... False False
│         * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
│         * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
│         * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
│       Data variables:
│           dx                     float64 8B ...
│           invert_with_trapezoid  bool 1B ...
│           is_last                bool 1B ...
└── Group: /inversion_output
        Dimensions:                (flux: 50, flux_a0: 50, hgt: 50, is_rectangular: 50,
                                    is_trapezoid: 50, slope_angle: 50, thick: 50,
                                    volume: 50, width: 50)
        Coordinates:
          * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
          * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
          * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
          * is_rectangular         (is_rectangular) bool 50B False False ... False False
          * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
          * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
          * thick                  (thick) float64 400B 33.17 35.88 ... 38.42 27.32
          * volume                 (volume) float64 400B 2.5e+06 2.068e+06 ... 1.257e+06
          * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
        Data variables:
            dx                     float64 8B ...
            invert_with_trapezoid  bool 1B ...
            is_last                bool 1B ...

In [11]:
stream_cube.inversion_input

<xarray.DataTree 'inversion_input'>
Group: /inversion_input
    Dimensions:                (flux: 50, flux_a0: 50, hgt: 50, is_rectangular: 50,
                                is_trapezoid: 50, slope_angle: 50, width: 50)
    Coordinates:
      * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
      * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
      * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
      * is_rectangular         (is_rectangular) bool 50B False False ... False False
      * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
      * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
      * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
    Data variables:
        dx                     float64 8B ...
        invert_with_trapezoid  bool 1B ...
        is_last                bool 1B ...